# Использование тактик и стратегий для решения задач

In [1]:
# Установка зависимостей для воспроизведения ноутбука
!pip install z3-solver

In [2]:
# Импорт z3
from z3 import *

## Примеры использования тактик

### Задача №1 - Сложения векторов в ноль
В качестве входных данных - n целочисленных векторов одинаковой размерности. Заранее неизвестна ни размерность, ни значения векторов. Необходимо выдать ответ - существует ли множество векторов, сумма которых равна нулевому вектору.

In [9]:

# Function for solving
def solve_zero_subset(vectors):
    if not vectors:
        return None

    n = len(vectors)
    d = len(vectors[0])

    if any(len(v) != d for v in vectors):
        raise ValueError("All vectors must have equal dimension")

    # Selection variables
    x = [Int(f"x_{i}") for i in range(n)]

    # Three concurrent solving pipelines
    linear = Then(
        'qflia',
        'fail-if-undecided',
    )

    general = Then(
        'smt',
        'fail-if-undecided',
    )

    sat_based = Then(
        'simplify',
        'normalize-bounds',
        'lia2pb',
        'pb2bv',
        'bit-blast',
        'sat',
        'fail-if-undecided',
    )

    portfolio = ParOr(
        linear,
        general,
        sat_based,
    ).solver()

    # Every vector is either selected or not
    for xi in x:
        portfolio.add(0 <= xi, xi <= 1)

    # Non-empty subset
    portfolio.add(Sum(x) >= 1)

    # Every coordinate of the sum must be zero
    for j in range(d):
        portfolio.add(
            Sum(x[i] * vectors[i][j] for i in range(n)) == 0
        )

    result = portfolio.check()

    if result == unsat:
        return None

    if result == unknown:
        raise RuntimeError(portfolio.reason_unknown())

    model = portfolio.model()

    indices = [
        i for i in range(n)
        if model.eval(x[i], model_completion=True).as_long() == 1
    ]

    assert indices
    assert all(
        sum(vectors[i][j] for i in indices) == 0
        for j in range(d)
    )

    return indices


# INPUT DATA
vectors =  [
        [
          -695157,
          939382,
          -598203,
          -72184,
          -558363,
          -764046,
          -193381,
          740440,
          405477,
          582928,
          -229971,
          431831
        ],
        [
          -639515,
          -571053,
          -546654,
          514745,
          -577162,
          -116175,
          178057,
          -797469,
          -174654,
          -114261,
          -743480,
          -719264
        ],
        [
          441655,
          -5811,
          113395,
          305340,
          578109,
          228409,
          -604235,
          -647152,
          10642,
          -728727,
          781487,
          -449068
        ],
        [
          -999988,
          825762,
          206124,
          -332215,
          683096,
          -960681,
          -99941,
          768237,
          469096,
          146021,
          772878,
          285959
        ],
        [
          -436613,
          334649,
          965259,
          613010,
          893052,
          -741635,
          -944586,
          -932111,
          620984,
          806032,
          337067,
          -579731
        ],
        [
          -31158,
          60164,
          -923496,
          -432947,
          -352755,
          -440910,
          782168,
          615482,
          -762599,
          -989655,
          75950,
          -856127
        ],
        [
          -42255,
          938903,
          -134441,
          -793471,
          332006,
          131439,
          453958,
          672042,
          -508755,
          726355,
          922907,
          277458
        ],
        [
          -103138,
          -957348,
          -517794,
          -478134,
          942044,
          -123884,
          787732,
          590189,
          794802,
          -345485,
          -281809,
          609008
        ],
        [
          713965,
          94468,
          283572,
          405394,
          442488,
          107768,
          478924,
          480024,
          614490,
          -311807,
          -203442,
          135325
        ],
        [
          -455821,
          528716,
          792929,
          -747897,
          669920,
          574819,
          219693,
          100254,
          516355,
          121257,
          -949255,
          368891
        ],
        [
          620241,
          28005,
          3957,
          673761,
          -570581,
          219302,
          -584250,
          -962222,
          605315,
          -377613,
          879809,
          727401
        ],
        [
          894926,
          -264165,
          497355,
          367401,
          -775801,
          945689,
          -732948,
          -72877,
          90821,
          -458588,
          -554275,
          -112116
        ],
        [
          652549,
          61574,
          710551,
          -190885,
          360581,
          -634750,
          -600594,
          991595,
          -48673,
          122973,
          -266706,
          -454455
        ],
        [
          92560,
          93654,
          -858668,
          -459266,
          892902,
          -769269,
          -762535,
          671519,
          726269,
          -749301,
          -832777,
          -61735
        ],
        [
          -753249,
          -853568,
          28285,
          -769222,
          -57075,
          208804,
          -67806,
          632574,
          -571000,
          -512815,
          989670,
          47735
        ],
        [
          852028,
          -351852,
          971387,
          119121,
          75065,
          -197659,
          453795,
          -554885,
          726598,
          -552787,
          -456557,
          462033
        ],
        [
          -801123,
          -474085,
          -609307,
          -833849,
          666138,
          -973919,
          285761,
          -115559,
          -512205,
          -551944,
          -452376,
          -33019
        ],
        [
          -220622,
          -65269,
          501744,
          -969906,
          576058,
          550085,
          -543335,
          836419,
          827776,
          -54756,
          -845134,
          -277330
        ],
        [
          -69165,
          805884,
          810625,
          -13443,
          -467447,
          633733,
          407468,
          -591331,
          224209,
          367347,
          728379,
          146794
        ],
        [
          -799387,
          76969,
          444981,
          -217150,
          96924,
          823032,
          398525,
          -874023,
          236511,
          -902569,
          457264,
          -703453
        ],
        [
          422921,
          40504,
          -563092,
          238388,
          387364,
          530835,
          -767566,
          461058,
          593852,
          18018,
          -318814,
          -993826
        ],
        [
          -544702,
          257981,
          578755,
          -579671,
          -325979,
          979138,
          -460317,
          -787330,
          684981,
          696814,
          755698,
          -564554
        ],
        [
          698329,
          -259141,
          -321815,
          -448051,
          7066,
          -247339,
          125159,
          801209,
          923259,
          525170,
          418183,
          -787586
        ],
        [
          -210706,
          -379261,
          -206604,
          -390490,
          373196,
          -158372,
          334286,
          -899745,
          -723080,
          -642021,
          -311311,
          -646882
        ],
        [
          -110375,
          791815,
          680616,
          -421116,
          -108862,
          948797,
          165548,
          988745,
          272536,
          -748917,
          -180875,
          -728104
        ],
        [
          -967521,
          -774998,
          769626,
          247959,
          940908,
          -349174,
          -78742,
          626881,
          -987924,
          598399,
          -580265,
          -415244
        ],
        [
          -46828,
          -505423,
          515551,
          32271,
          -586629,
          -16945,
          -908157,
          491993,
          169055,
          -125397,
          -499053,
          -529575
        ],
        [
          -997558,
          941302,
          329921,
          979587,
          337160,
          121376,
          426634,
          -319994,
          -245080,
          -28723,
          -490972,
          843962
        ],
        [
          301901,
          67859,
          571147,
          -55882,
          -603027,
          -890219,
          940138,
          -908284,
          -422277,
          -95853,
          -170224,
          907779
        ],
        [
          -435919,
          132977,
          -34026,
          580787,
          -577680,
          664673,
          -817909,
          -949423,
          -890649,
          -417936,
          655187,
          -991351
        ],
        [
          964906,
          596965,
          760802,
          -85764,
          -533955,
          -409277,
          -589012,
          -379273,
          349799,
          -812967,
          770796,
          -388320
        ],
        [
          995254,
          24349,
          655391,
          -32455,
          -850084,
          -640905,
          77450,
          -804826,
          101036,
          -291860,
          -398270,
          945667
        ],
        [
          -675728,
          -591191,
          -983683,
          -384951,
          -53680,
          -93960,
          -231439,
          -988743,
          -664352,
          41322,
          -335020,
          161908
        ],
        [
          -49961,
          -389005,
          -795637,
          -16413,
          -983107,
          953213,
          -552431,
          172858,
          -740334,
          -904912,
          716227,
          -60579
        ],
        [
          201938,
          -541407,
          -906100,
          833639,
          -579670,
          -35313,
          135269,
          947037,
          -429625,
          700741,
          -229577,
          418511
        ],
        [
          598078,
          -909546,
          266307,
          -912993,
          -767347,
          -790284,
          886596,
          -142143,
          93165,
          -756548,
          35749,
          44719
        ],
        [
          -16045,
          -643778,
          -583467,
          -283200,
          839042,
          149715,
          -751554,
          673046,
          60222,
          -728791,
          -326279,
          -618217
        ],
        [
          -520926,
          -495447,
          -769173,
          978896,
          642647,
          212231,
          993432,
          -247237,
          316665,
          -779087,
          405852,
          -999725
        ],
        [
          -148688,
          -515297,
          943180,
          813823,
          507061,
          586159,
          446771,
          -499171,
          309516,
          378456,
          -965130,
          758795
        ],
        [
          874794,
          -676873,
          96593,
          -622994,
          687991,
          -435456,
          -575335,
          572400,
          -533039,
          207322,
          -470237,
          -951721
        ],
        [
          -940506,
          1960476,
          -80506,
          -263754,
          1427591,
          742675,
          -648185,
          1062293,
          341497,
          448996,
          723182,
          -995613
        ],
        [
          690499,
          -913979,
          -218030,
          -169391,
          -785120,
          843635,
          465249,
          -261356,
          489777,
          180606,
          718692,
          525976
        ],
        [
          -183853,
          73721,
          -431157,
          582469,
          985593,
          -708870,
          597381,
          324802,
          575751,
          640633,
          859531,
          759735
        ],
        [
          679775,
          -128736,
          -340952,
          751267,
          -506086,
          -74449,
          63789,
          731190,
          -878299,
          5293,
          -488021,
          -695380
        ],
        [
          298580,
          601021,
          612510,
          911667,
          -367358,
          -329215,
          -789461,
          -31580,
          -196492,
          -157456,
          -131770,
          -752827
        ],
        [
          131580,
          -711456,
          -175772,
          733209,
          60744,
          57302,
          756517,
          649639,
          -739754,
          -124751,
          852730,
          -994790
        ],
        [
          -119152,
          395218,
          -725408,
          362789,
          957831,
          -512479,
          -373608,
          586722,
          311461,
          891015,
          364712,
          -822765
        ],
        [
          595289,
          -532821,
          -226778,
          295020,
          26562,
          -396485,
          -564169,
          776574,
          727354,
          975631,
          35683,
          -899422
        ],
        [
          -478280,
          -433231,
          783185,
          647032,
          924195,
          -958947,
          237753,
          -243274,
          -874626,
          664052,
          29043,
          -776186
        ],
        [
          -11057,
          366899,
          517130,
          -315759,
          -624332,
          154712,
          -425220,
          -696928,
          -875276,
          909488,
          699582,
          -772755
        ]
      ]

solve_zero_subset(vectors)

[30, 32, 34, 35, 38, 40]

### Задача №2 - люди динозавры
На вход подаётся произвольное количество текстовых утверждений о родственных связях между различными видами животных.

Утверждения могут описывать прямое родство, отношения предков и потомков, а также отсутствие определённых родственных связей. Некоторые утверждения относятся к одной цепочке происхождения, другие образуют независимые группы или вообще не содержат полезной информации.

Необходимо на основании предоставленных сведений определить, являются ли динозавры предками людей - вывести True или False.


In [17]:
from __future__ import annotations

import argparse
from collections import deque
from dataclasses import dataclass
import json
from pathlib import Path
import re
from typing import Iterable


STATEMENT = re.compile(
    r"^(?P<left>[a-z][a-z\s-]*?)\s+"
    r"(?:(?:is|are)\s+)?"
    r"(?P<not>not\s+)?"
    r"(?:(?:a|an)\s+)?"
    r"(?P<kind>parents?|ancestors?|ansestors?|child|children|childs)\s+"
    r"(?:of|for)\s+"
    r"(?P<right>[a-z][a-z\s-]*?)\s*[.!?]?$",
    re.IGNORECASE,
)


def canonical(value: str) -> str:
    return " ".join(value.lower().strip().split())


def parse_phrases(phrases: Iterable[str]):
    """Return positive ancestry edges, negative facts, and ignored phrases.

    Positive parent / child assertions become ancestry edges; only a known
    positive ancestry chain can prove the requested relationship.
    Negative facts are kept for inspection, but do not imply positive edges.
    The task assumes the input is logically consistent.
    """
    edges: set[tuple[str, str]] = set()
    negatives: list[tuple[str, str, str]] = []
    ignored: list[str] = []
    animals: set[str] = set()

    for phrase in phrases:
        match = STATEMENT.fullmatch(phrase.strip())
        if match is None:
            ignored.append(phrase)
            continue

        left, right = canonical(match['left']), canonical(match['right'])
        is_negative = match['not'] is not None
        kind = match['kind'].lower()
        kind = ('ancestor' if kind.startswith(('ancestor', 'ansestor'))
                else 'parent' if kind.startswith('parent') else 'child')

        # "not child of A or B" means two negative assertions.
        # Positive disjunctions are outside the supported grammar.
        targets = [canonical(s) for s in re.split(r'\s+or\s+', right)]
        if not is_negative and len(targets) != 1:
            ignored.append(phrase)
            continue

        for other in targets:
            animals.update((left, other))
            if is_negative:
                negatives.append((kind, left, other))
            elif kind == 'child':
                edges.add((other, left))
            else:
                edges.add((left, other))

    return animals, edges, negatives, ignored


def adjacency(edges: set[tuple[str, str]]) -> dict[str, set[str]]:
    result: dict[str, set[str]] = {}
    for parent, child in edges:
        result.setdefault(parent, set()).add(child)
    return result


def bfs_path(start, goal, graph):
    """Shortest path in a directed graph, or None."""
    pending = deque([start])
    previous = {start: None}
    while pending:
        node = pending.popleft()
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = previous[node]
            return path[::-1]
        for neighbor in sorted(graph.get(node, ())):
            if neighbor not in previous:
                previous[neighbor] = node
                pending.append(neighbor)
    return None


def quotient_graph(blocks, edges):
    """Over-approximate graph: vertices in the same block are merged."""
    block_of = {name: i for i, block in enumerate(blocks) for name in block}
    abstract_edges = {
        (block_of[a], block_of[b])
        for a, b in edges
        if block_of[a] != block_of[b]
    }
    return block_of, abstract_edges


def abstract_path_python(start, end, block_count, abstract_edges):
    g = adjacency(abstract_edges)
    return bfs_path(start, end, g)


def abstract_path_z3(start, end, block_count, abstract_edges):
    """Ask Z3 for the shortest abstract path by incremental unrolling.

    Any reachable path has a simple representative of at most B-1 edges,
    where B is the current number of abstract blocks.
    """
    try:
        from z3 import And, Int, Or, Solver, sat, unsat
    except ImportError as exc:
        raise RuntimeError('Install z3-solver to use --engine z3') from exc

    edges = sorted(abstract_edges)
    solver = Solver()
    states = [Int('block_0')]
    solver.add(states[0] == start)

    for length in range(block_count):
        solver.push()
        solver.add(states[-1] == end)
        result = solver.check()
        if result == sat:
            model = solver.model()
            path = [model.eval(x).as_long() for x in states]
            solver.pop()
            return path
        solver.pop()
        if result != unsat:
            raise RuntimeError(f'Z3 returned {result}')
        if length == block_count - 1 or not edges:
            break
        nxt = Int(f'block_{length + 1}')
        solver.add(Or(*[
            And(states[-1] == u, nxt == v)
            for u, v in edges
        ]))
        states.append(nxt)
    return None


def reachable_inside(seeds, block, graph):
    """Exact reachability without leaving this abstract block."""
    allowed = set(block)
    reached = set(seeds)
    q = deque(seeds)
    while q:
        u = q.popleft()
        for v in graph.get(u, ()):
            if v in allowed and v not in reached:
                reached.add(v)
                q.append(v)
    return reached


def validate_and_refine(path, blocks, graph, source, target):
    """Concretize an abstract path.

    Returns (is_real, split_block_id, reachable_subset).
    On a spurious path, the reachable subset is a nonempty proper subset of
    the identified block. Splitting on it eliminates this spurious mixing.
    """
    current = {source}
    for step, block_id in enumerate(path):
        current = reachable_inside(current, blocks[block_id], graph)
        if step == len(path) - 1:
            if target in current:
                return True, None, None
            return False, block_id, current

        following = set(blocks[path[step + 1]])
        successors = {
            v for u in current
            for v in graph.get(u, ()) if v in following
        }
        if not successors:
            return False, block_id, current
        current = successors

    raise AssertionError('An abstract path must have at least one block')


def split_block(blocks, which, subset):
    block = set(blocks[which])
    a = tuple(sorted(subset))
    b = tuple(sorted(block - set(subset)))
    if not a or not b:
        raise AssertionError('Refinement must strictly split a block')
    return blocks[:which] + [a, b] + blocks[which + 1:]


@dataclass
class CegarResult:
    answer: bool
    refinements: int
    abstract_checks: int
    witness_path: list[str] | None
    ignored_phrases: list[str]
    negative_fact_count: int


def solve_ancestry(phrases: list[str], *, source='dinosaurs',
                   target='humans', engine='python', verbose=False) -> CegarResult:
    """Check positive ancestry entailment by genuine partition-refinement CEGAR."""
    source, target = canonical(source), canonical(target)
    animals, edges, negatives, ignored = parse_phrases(phrases)
    animals.update((source, target))
    graph = adjacency(edges)

    # Intentionally lossy abstraction: merge names beginning with the same
    # letter. Different concrete animals can appear as a single abstract node.
    buckets: dict[str, list[str]] = {}
    for animal in sorted(animals):
        buckets.setdefault(animal[0], []).append(animal)
    blocks = [tuple(group) for _, group in sorted(buckets.items())]
    checks = 0
    refinements = 0

    for _ in range(len(animals) + 1):
        block_of, abstract_edges = quotient_graph(blocks, edges)
        if engine == 'python':
            abstract_path = abstract_path_python(
                block_of[source], block_of[target], len(blocks), abstract_edges
            )
        elif engine == 'z3':
            abstract_path = abstract_path_z3(
                block_of[source], block_of[target], len(blocks), abstract_edges
            )
        else:
            raise ValueError("engine must be 'python' or 'z3'")

        checks += 1
        if abstract_path is None:
            if verbose:
                print(f'  [{checks}] No abstract path: FALSE')
            return CegarResult(False, refinements, checks, None, ignored, len(negatives))

        real, bad_block, reachable = validate_and_refine(
            abstract_path, blocks, graph, source, target
        )
        if real:
            witness = bfs_path(source, target, graph)
            assert witness is not None
            if verbose:
                print(f'  [{checks}] Real path: {" -> ".join(witness)}')
            return CegarResult(True, refinements, checks, witness, ignored, len(negatives))

        if verbose:
            print(f'  [{checks}] Spurious path: '
                  f'{[list(blocks[i]) for i in abstract_path]}; '
                  f'split {list(blocks[bad_block])} into {sorted(reachable)} + others')
        blocks = split_block(blocks, bad_block, reachable)
        refinements += 1

    raise AssertionError('CEGAR did not converge despite strict refinement')

# Simple examples
# phrases = [
#     "Dinosaurs are ancestors of mammals.",
#     "Mammals are ancestors of humans.",
#     "Pigs are ancestors of bats.",
# ]


phrases = [
    "Dinosaurs are ancestors of old reptiles.",
    "Old reptiles are ancestors of birds.",
    "Dinosaurs are parents of marine reptiles.",
    "Marine reptiles are ancestors of whales.",
    "Pigs are ancestors of bats.",
    "Bats are ancestors of felines.",
    "Felines are ancestors of humans.",
    "Gorillas are children of felines.",
    "Birds are not ancestors of humans.",
    "Lizards are not children of gorillas.",
    "Some things are completely irrelevant.",
    "Dinosaurs are ancestors of old reptiles.",
    "Owls are ancestors of humans.",
    "Dinosaurs are ancestors of cave reptiles.",
    "Cats are ancestors of humans."
]

result = solve_ancestry(
    phrases,
    engine="python",
    verbose=True,
)

print(result.answer)
print(result.refinements)
print(result.witness_path)

  [1] Spurious path: [['dinosaurs'], ['cats', 'cave reptiles'], ['humans']]; split ['cats', 'cave reptiles'] into ['cave reptiles'] + others
  [2] Spurious path: [['dinosaurs'], ['old reptiles', 'owls'], ['humans']]; split ['old reptiles', 'owls'] into ['old reptiles'] + others
  [3] Spurious path: [['dinosaurs'], ['old reptiles'], ['bats', 'birds'], ['felines'], ['humans']]; split ['bats', 'birds'] into ['birds'] + others
  [4] No abstract path: FALSE
False
3
None


### Задача №3 - Орбита в спутниках?

Необходимо разместить n спутников вокргу земли. Для каждого спутника задано множество допустимых для него орбит.
Каждый спутник должен занимать ровно одну орбиту. Расстояние между номерами орбит любых ближайших спутников должно быть не менее 2 км.

Определить, можно ли разместить все спутники, и найти подходящее распределение.

In [18]:
DATA = {
    'Hubble': [2, 3],
    'Terra':  [1, 4],
    'Astra':  [5, 6],
    'Luna':   [7, 8],
    'Solar':  [9, 10],
}


class OrbitPropagator(UserPropagateBase):
    def __init__(self, solver, choices):
        super().__init__(solver)
        self.choices = choices  # (satellite, orbit) -> Bool
        self.propagations = 0
        self.add_fixed(self.on_fixed)
        for term in choices.values():
            self.add(term)

    def push(self):
        pass  # no custom mutable search state

    def pop(self, num_scopes):
        pass

    def fresh(self, ctx):
        raise NotImplementedError('This example uses one solver context')

    def on_fixed(self, term, value):
        if not is_true(value):
            return
        selected = next(key for key, var in self.choices.items() if var.eq(term))
        satellite, orbit = selected
        for (other, other_orbit), var in self.choices.items():
            if other != satellite and abs(orbit - other_orbit) < 2:
                self.propagate(Not(var), [term])
                self.propagations += 1


def solve(data):
    solver = Solver()
    choices = {(sat, orbit): Bool(f'orbit_{sat}_{orbit}')
               for sat, domain in data.items() for orbit in domain}
    theory = OrbitPropagator(solver, choices)

    for sat, domain in data.items():
        solver.add(PbEq([(choices[sat, p], 1) for p in domain], 1))

    status = solver.check()
    if status == unknown:
        raise RuntimeError(solver.reason_unknown())
    if status == unsat:
        return None, theory.propagations

    model = solver.model()
    result = {sat: next(p for p in domain
                        if is_true(model.eval(choices[sat, p], model_completion=True)))
              for sat, domain in data.items()}
    values = list(result.values())
    assert all(abs(a - b) >= 2 for i, a in enumerate(values) for b in values[i + 1:])
    return result, theory.propagations



result, count = solve(DATA)
print('SAT:', result is not None)
print('Orbits:', result)
print('Custom propagations:', count)


SAT: True
Orbits: {'Hubble': 3, 'Terra': 1, 'Astra': 5, 'Luna': 7, 'Solar': 9}
Custom propagations: 5


### Задача №4 - Идеальный резонансо-баланс 
Несколько спутников вращаются вокруг Юпитера. У каждого есть ограниченный набор допустимых периодов обращения в сутах. Некоторые пары спутников должны находиться в резонансе 1:2 - пока первый совершает два оборота, второй совершает один.

Необходимо определить, существует ли набор периодов, при котором выполняются все условия резонанса 1:2

In [29]:
DATA = {
    'periods': {
        'Io':       [2, 3],
        'Europa':   [4, 6],
        'Ganymede': [8, 12],
        'Callisto': [15, 16],
    },
    'double_period': [
        ('Io', 'Europa'),
        ('Europa', 'Ganymede'),
        ('Ganymede', 'Callisto'),
    ],
}


class ResonancePropagator(UserPropagateBase):
    def __init__(self, solver, choices, links, periods):
        super().__init__(solver)
        self.choices = choices  # (moon, period) -> Bool
        self.rules = {}
        self.propagations = 0
        self.conflicts = 0

        for a, b in links:
            for p in periods[a]:
                self.rules.setdefault((a, p), []).append((b, 2 * p))
            for q in periods[b]:
                self.rules.setdefault((b, q), []).append((a, q // 2 if q % 2 == 0 else None))

        self.add_fixed(self.on_fixed)
        for term in choices.values():
            self.add(term)

    def push(self):
        pass

    def pop(self, num_scopes):
        pass

    def fresh(self, ctx):
        raise NotImplementedError('This example uses one solver context')

    def on_fixed(self, term, value):
        if not is_true(value):
            return
        selected = next(key for key, var in self.choices.items() if var.eq(term))
        for target in self.rules.get(selected, []):
            consequence = self.choices.get(target)
            if consequence is None:
                self.conflicts += 1
                self.conflict([term])
                return
            self.propagations += 1
            self.propagate(consequence, [term])


def solve(data):
    periods, links = data['periods'], data['double_period']
    solver = Solver()
    choices = {(moon, p): Bool(f'period_{moon}_{p}')
               for moon, domain in periods.items() for p in domain}
    theory = ResonancePropagator(solver, choices, links, periods)

    for moon, domain in periods.items():
        solver.add(PbEq([(choices[moon, p], 1) for p in domain], 1))

    status = solver.check()
    if status == unknown:
        raise RuntimeError(solver.reason_unknown())
    if status == unsat:
        return None, theory

    model = solver.model()
    result = {moon: next(p for p in domain
                         if is_true(model.eval(choices[moon, p], model_completion=True)))
              for moon, domain in periods.items()}
    assert all(result[b] == 2 * result[a] for a, b in links)
    return result, theory



result, theory = solve(DATA)
print('SAT:', result is not None)
print('Periods:', result)
print('Custom propagations:', theory.propagations)
print('Custom conflicts:', theory.conflicts)

SAT: True
Periods: {'Io': 2, 'Europa': 4, 'Ganymede': 8, 'Callisto': 16}
Custom propagations: 6
Custom conflicts: 1


### Задача №5 - Aвеню UnSat
Космический зонд должен добраться от Земли до Нептуна, используя ограниченный запас топлива и гравитационные манёвры возле других планет. Путешествие между двумя планетами требует определённого количества топлива. Посещение некоторых планет даёт бонус к запасу топлива за счёт гравитационного ускорения, а каждую доступную планету разрешено посетить только один раз.

Необходимо найти допустимый маршрут до Нептуна, не превышая запас доступного топлива

In [25]:
DATA = {
    'planets': ['Earth', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune'],
    'flights': [
        ('Earth', 'Venus', 9), ('Earth', 'Mars', 7), ('Earth', 'Jupiter', 15),
        ('Venus', 'Mars', 4), ('Venus', 'Jupiter', 9),
        ('Mars', 'Jupiter', 6), ('Mars', 'Saturn', 12),
        ('Jupiter', 'Saturn', 6), ('Jupiter', 'Uranus', 10),
        ('Saturn', 'Uranus', 5), ('Saturn', 'Neptune', 13),
        ('Uranus', 'Neptune', 7),
    ],
    'gravity_bonus': {'Venus': 1, 'Jupiter': 3, 'Saturn': 2, 'Uranus': 2},
    'fuel': 25,
}


class RoutePropagator(UserPropagateBase):
    def __init__(self, solver, edges, planets, flights, costs, budget):
        super().__init__(solver)
        self.edges, self.planets, self.flights = edges, planets, flights
        self.costs, self.budget = costs, budget
        self.assignments = {}  # edge index -> True/False
        self.trail, self.marks = [], []
        self.conflicts = 0
        self.add_fixed(self.on_fixed)
        for x in edges:
            self.add(x)

    def push(self):
        self.marks.append(len(self.trail))

    def pop(self, n):
        for _ in range(n):
            cut = self.marks.pop()
            for i in reversed(self.trail[cut:]):
                del self.assignments[i]
            del self.trail[cut:]

    def fresh(self, ctx):
        raise NotImplementedError('This example uses one solver context')

    def on_fixed(self, term, value):
        index = next(i for i, x in enumerate(self.edges) if x.eq(term))
        if index not in self.assignments:
            self.trail.append(index)
        self.assignments[index] = is_true(value)

        # Optimistic shortest path: all not-yet-forbidden flights remain available.
        best = {p: float('inf') for p in self.planets}
        best[self.planets[0]] = 0
        for planet in self.planets:
            for i, (a, b, _) in enumerate(self.flights):
                if a == planet and self.assignments.get(i) is not False:
                    best[b] = min(best[b], best[a] + self.costs[i])

        if best[self.planets[-1]] > self.budget:
            # Sound explanation: at least one currently forbidden edge must reopen.
            blockers = [self.edges[i] for i, flag in self.assignments.items() if not flag]            
            self.conflicts += 1
            self.conflict(blockers)


def solve(data):
    planets, flights = data['planets'], data['flights']
    fuel, bonus = data['fuel'], data['gravity_bonus']
    rank = {p: i for i, p in enumerate(planets)}
    assert all(rank[a] < rank[b] for a, b, _ in flights), 'Flights must form a DAG'
    costs = [c - bonus.get(b, 0) for _, b, c in flights]
    assert all(c >= 0 for c in costs), 'No negative net fuel costs'

    solver = Solver()
    edges = [Bool(f'flight_{i}') for i in range(len(flights))]
    theory = RoutePropagator(solver, edges, planets, flights, costs, fuel)

    for planet in planets:
        incoming = [edges[i] for i, (_, b, _) in enumerate(flights) if b == planet]
        outgoing = [edges[i] for i, (a, _, _) in enumerate(flights) if a == planet]
        in_count = Sum([If(x, 1, 0) for x in incoming])
        out_count = Sum([If(x, 1, 0) for x in outgoing])
        if planet == planets[0]:
            solver.add(in_count == 0, out_count == 1)
        elif planet == planets[-1]:
            solver.add(in_count == 1, out_count == 0)
        else:
            solver.add(in_count == out_count, in_count <= 1)

    # SMT guarantees the budget; custom shortest-path reasoning prunes impossible branches.
    solver.add(Sum([If(e, costs[i], 0) for i, e in enumerate(edges)]) <= fuel)
    status = solver.check()
    if status == unknown:
        raise RuntimeError(solver.reason_unknown())
    if status == unsat:
        return None, theory.conflicts

    model = solver.model()
    used = {a: (b, costs[i]) for i, (a, b, _) in enumerate(flights)
            if is_true(model.eval(edges[i], model_completion=True))}
    route, spent = [planets[0]], 0
    while route[-1] != planets[-1]:
        b, cost = used[route[-1]]
        route.append(b)
        spent += cost
    assert spent <= fuel and len(route) == len(set(route))
    return {'route': route, 'fuel_used': spent, 'fuel_limit': fuel}, theory.conflicts



result, conflicts = solve(DATA)
print('SAT:', result is not None)
print('Mission:', result)
print('Custom conflicts:', conflicts)

SAT: True
Mission: {'route': ['Earth', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune'], 'fuel_used': 24, 'fuel_limit': 25}
Custom conflicts: 1
